In [21]:
# 1. 현재 Notebook 환경 확인

import sys
from pathlib import Path

print("Python:", sys.version)
print("Executable:", sys.executable)
print("현재 위치:", Path.cwd())

Python: 3.12.10 (v3.12.10:0cc81280367, Apr  8 2025, 08:46:59) [Clang 13.0.0 (clang-1300.0.29.30)]
Executable: /Users/jee/Desktop/sso/disclosure-analyst-agent/.venv/bin/python
현재 위치: /Users/jee/Desktop/sso/disclosure-analyst-agent/notebooks


In [22]:
# 2. 프로젝트 / 데이터 경로 설정

PROJECT_ROOT = Path("..").resolve()
CORPUS_DIR = PROJECT_ROOT / "data" / "raw" 

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CORPUS_DIR:", CORPUS_DIR)
print("Corpus 존재:", CORPUS_DIR.exists())

PROJECT_ROOT: /Users/jee/Desktop/sso/disclosure-analyst-agent
CORPUS_DIR: /Users/jee/Desktop/sso/disclosure-analyst-agent/data/raw
Corpus 존재: True


In [23]:
UNIVERSE_PATH = PROJECT_ROOT / "data"  / "universe.csv"
MANIFEST_PATH = PROJECT_ROOT / "data"  / "manifest.jsonl"
RAW_DIR = PROJECT_ROOT / "data"  / "raw"

In [24]:
import pandas as pd

universe = pd.read_csv(
    UNIVERSE_PATH,
    dtype={
        "corp_code": str,
        "stock_code": str,
    },
)

manifest = pd.read_json(
    MANIFEST_PATH,
    lines=True,
    dtype={
        "corp_code": str,
        "stock_code": str,
    },
)

print("universe:", universe.shape)
print("manifest:", manifest.shape)

# display(universe.head())
# display(manifest.head())
print(universe.info())


universe: (70, 17)
manifest: (4204, 19)
<class 'pandas.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   corp_code      70 non-null     str  
 1   stock_code     70 non-null     str  
 2   corp_name      70 non-null     str  
 3   listed_name    70 non-null     str  
 4   corp_eng_name  70 non-null     str  
 5   market         70 non-null     str  
 6   industry       70 non-null     str  
 7   sector_no      70 non-null     int64
 8   sector         70 non-null     str  
 9   listing_date   70 non-null     str  
 10  fiscal_month   70 non-null     str  
 11  market_cap     70 non-null     int64
 12  n_periodic     70 non-null     int64
 13  n_major        70 non-null     int64
 14  n_exchange     70 non-null     int64
 15  n_holding      70 non-null     int64
 16  note           5 non-null      str  
dtypes: int64(6), str(11)
memory usage: 9.4 KB
None


In [25]:
!pip install beautifulsoup4
from bs4 import BeautifulSoup


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [26]:
from pathlib import Path
from bs4 import BeautifulSoup
from charset_normalizer import from_bytes
import re


# ============================================================
# 1. 고려아연 exchange 폴더
# ============================================================

company_dir = Path(
    "/Users/jee/Desktop/sso/disclosure-analyst-agent/"
    "data/raw/exchange/고려아연"
)


# ============================================================
# 2. XML 파일 하나 찾기
# ============================================================

xml_files = sorted(company_dir.rglob("*.xml"))

print(f"XML 파일 수: {len(xml_files)}")

if not xml_files:
    raise FileNotFoundError("XML 파일을 찾을 수 없습니다.")

# 일단 첫 번째 파일
xml_path = xml_files[0]

print("선택된 파일:")
print(xml_path)


# ============================================================
# 3. 인코딩 자동 감지
# ============================================================

raw = xml_path.read_bytes()

detected = from_bytes(raw).best()

if detected is None:
    raise ValueError("인코딩 감지 실패")

print("\nDetected encoding:", detected.encoding)

html = str(detected)


# ============================================================
# 4. HTML 파싱
# ============================================================

soup = BeautifulSoup(html, "html.parser")


# CSS / script 등 불필요한 요소 제거
for tag in soup(["style", "script", "noscript"]):
    tag.decompose()


# ============================================================
# 5. 제목
# ============================================================

title = None

if soup.title:
    title = soup.title.get_text(" ", strip=True)

print("\nTITLE")
print("=" * 100)
print(title)


# ============================================================
# 6. 테이블을 사람이 읽기 좋은 형태로 추출
# ============================================================

lines = []

for table_idx, table in enumerate(soup.find_all("table")):

    rows = []

    for tr in table.find_all("tr"):

        cells = []

        for cell in tr.find_all(["th", "td"], recursive=False):

            text = cell.get_text(
                " ",
                strip=True
            )

            text = text.replace("\xa0", " ")
            text = re.sub(r"\s+", " ", text).strip()

            if text:
                cells.append(text)

        if cells:
            rows.append(cells)

    if not rows:
        continue

    lines.append(f"\n[TABLE {table_idx}]")

    for row in rows:
        lines.append(" | ".join(row))


# ============================================================
# 7. 최종 clean text
# ============================================================

clean_text = "\n".join(lines)


# ============================================================
# 8. 출력
# ============================================================

print("\n\nCLEAN TEXT")
print("=" * 100)
print(clean_text)

ModuleNotFoundError: No module named 'charset_normalizer'